# Report-Aligned Audit Notebook

This notebook is the audit entry point for reproducing the submitted DR classification and XAI pipeline artifacts.


In [ ]:
# Optional (Colab): Drive shortcut other team members
# 1. In Google Drive, open shared folder 'ITPG708Project'.
# 2. Click 'Add shortcut to Drive' -> choose 'My Drive'.
# 3. If Colab says Drive is already mounted, use force_remount=True and rerun this cell.

from pathlib import Path
import os

def _is_project_root(candidate: Path) -> bool:
    return (
        (candidate / 'requirements.txt').exists()
        and (candidate / 'configs' / 'base.yaml').exists()
        and (candidate / 'src' / 'data.py').exists()
    )

try:
    from google.colab import drive

    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/ITPG708Project')
    if not _is_project_root(BASE_DIR):
        raise FileNotFoundError(
            "Project folder not found at /content/drive/MyDrive/ITPG708Project.\n"
            "Add shared-folder shortcut to My Drive, then remount and rerun this cell."
        )
    os.chdir(BASE_DIR)
    print('Colab working directory:', os.getcwd())
except ImportError:
    print('Not running in Colab. Skip this cell and run Section 1 setup.')


## 1. Audit Setup and Configuration

Imports the project modules and sets up working paths, Matplotlib, warnings, seed, split, configuration, and optional cleanup controls.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    load_project_config,
    notebook_prepare_data_overview,
)
from src.train import (
    clean_generated_outputs,
    notebook_run_training,
    notebook_run_core_evaluation,
)
from src.xai_audit import (
    notebook_run_xai,
    notebook_load_xai_committee_summary,
    notebook_load_xai_advanced_audit,
    notebook_run_visual_review,
)
from src.xai_single import notebook_run_single_case_report

display(Markdown(f'**Project root:** `{PROJECT_ROOT}`'))


### Run Configuration

Defines the seed, evaluation split, config file, checkpoint reuse policy, and safe-mode switches used by downstream cells.


In [ ]:
CONFIG_PATH = str((PROJECT_ROOT / 'configs/base.yaml').resolve())
SEED = 1988
EVAL_SPLIT = 'test'
RUN_CLEAN_BEFORE_START = False
FORCE_RETRAIN = False
XAI_SAFE_MODE = False
SHOW_ADVANCED_XAI_AUDIT = False
XAI_VIS_SAFE_MODE = False
XAI_SINGLE_SAFE_MODE = False

cfg = load_project_config(CONFIG_PATH)
protocol = str(cfg.get('data', {}).get('protocol', 'default'))

display(Markdown(f'**Config:** `{CONFIG_PATH}`'))
display(Markdown(
    f'**Protocol:** `{protocol}` | **Seed:** `{SEED}` | **Split:** `{EVAL_SPLIT}` | '
    f'**Backbone:** `{cfg["training"].get("backbone", "resnet50")}`'
))

display(Markdown(
    f'**Benchmark split setup:** test_ratio={cfg["data"]["profile_test_ratio"]:.2f}, '
    f'val_within_train={cfg["data"]["profile_val_ratio_within_train"]:.2f}'
))

display(Markdown(
    f'**Preprocessing:** `{cfg.get("preprocessing", {}).get("enabled", True)}` | '
    f'**Loss:** `{cfg["training"].get("loss_name", "auto")}` | '
    f'**Weighted sampler:** `{cfg["training"].get("use_weighted_sampler", True)}` | '
    f'**Force retrain:** `{FORCE_RETRAIN}`'
))

display(Markdown('**Note:** If `RUN_CLEAN_BEFORE_START=True`, checkpoints are deleted first; retraining is expected even when `FORCE_RETRAIN=False`.'))


### Optional Output Reset

Optionally deletes stale generated outputs before rerunning. Leave disabled during audit unless a full rerun is intended.


In [ ]:
if RUN_CLEAN_BEFORE_START:
    clean_generated_outputs(CONFIG_PATH)
    print('Generated outputs cleaned.')
else:
    print('Cleanup skipped.')


## 2. Dataset and Preprocessing Pipeline

Loads the APTOS 2019 manifests, reports split counts and class distributions, and renders the dataset/preprocessing overview artifacts used by the report.


In [ ]:
data_overview = notebook_prepare_data_overview(cfg, seed=SEED, samples_per_split=3)
manifest_paths = data_overview['manifest_paths']
train_df = data_overview['train_df']
val_df = data_overview['val_df']
test_df = data_overview['test_df']

display(Markdown('### Manifest Files'))
display(pd.DataFrame([manifest_paths]).T.rename(columns={0: 'path'}))

display(Markdown('### Split Sizes'))
display(data_overview['split_summary'])

display(Markdown('### Overall Class Proportions'))
display(data_overview['overall_class_proportions'])

if len(data_overview['source_dataset_mix']):
    display(Markdown('### Source Dataset Mix (Roboflow vs APTOS)'))
    display(data_overview['source_dataset_mix'])
else:
    display(Markdown('### Source Dataset Mix'))
    display(Markdown('`source_dataset` column not found in manifests.'))

display(data_overview['class_count_fig'])
plt.close(data_overview['class_count_fig'])
display(Markdown('*Figure 5 in the report — Class distribution by split (counts). Training set has 2,800 images, validation 312, test 550.*'))

display(data_overview['class_proportion_fig'])
plt.close(data_overview['class_proportion_fig'])
display(Markdown('*Figure 6 in the report — Class distribution within each split (percentage). Class proportions are consistent across all three splits.*'))


## 3. Classification Model Training

Fine-tunes the EfficientNet-B4 backbone with focal loss, or reuses the cached checkpoint when the submitted weights already exist.


In [ ]:
training_out = notebook_run_training(
    cfg,
    seed=SEED,
    manifests=manifest_paths,
    force_retrain=FORCE_RETRAIN,
)
checkpoint_path = training_out['checkpoint_path']
RUN_ID = str(training_out['run_id'])

display(Markdown(f'Checkpoint path: `{checkpoint_path}`'))
display(Markdown(f'Run ID: `{RUN_ID}`'))
display(Markdown(
    f"**Checkpoint reuse:** `{training_out['reused_checkpoint']}` | "
    f"**Reason:** `{training_out['reuse_reason']}`"
))


## 4. Classification Model Evaluation

Runs inference on the test split and produces the headline metrics, per-class metrics, confusion matrix, calibration outputs, and related report tables.


In [ ]:
core_eval = notebook_run_core_evaluation(
    cfg,
    seed=SEED,
    split=EVAL_SPLIT,
)

display(Markdown(f'**Predictions used for evaluation:** `{core_eval["predictions_path"]}`'))
display(Markdown('### Evaluation Output Files'))
display(core_eval['eval_output_paths_df'])

display(Markdown('### Overall Metrics + Run Summary'))
display(core_eval['overall_summary_df'])
display(Markdown('*Table 3 in the report — Overall test-set metrics (APTOS 2019, 550 images, seed 1988).*'))

display(Markdown('### Per-Class Metrics'))
display(core_eval['per_class_df'])
display(Markdown('*Table 4 in the report — Per-class test metrics.*'))

## 5. XAI Methodology

Runs the Grad-CAM and SHAP explanation pipeline, applies mask-aware attribution correction, and writes the per-sample XAI audit outputs.


In [ ]:
xai_run = notebook_run_xai(
    cfg,
    seed=SEED,
    split=EVAL_SPLIT,
    manifests=manifest_paths,
    checkpoint=checkpoint_path,
    safe_mode=XAI_SAFE_MODE,
)
cfg_xai = xai_run['cfg_xai']
xai_outputs = xai_run['xai_outputs']
if xai_run['run_id']:
    RUN_ID = str(xai_run['run_id'])

display(Markdown(
    f"**XAI runtime settings:** safe_mode={xai_run['safe_mode']}, "
    f"device={xai_run['xai_runtime_settings']['device']}, "
    f"max_targets={xai_run['xai_runtime_settings']['max_targets']}, "
    f"shap_max_samples={xai_run['xai_runtime_settings']['shap_max_samples']}, "
    f"shap_background_size={xai_run['xai_runtime_settings']['shap_background_size']}, "
    f"log_every_samples={xai_run['xai_runtime_settings']['log_every_samples']}"
))

display(Markdown('### XAI Output Files'))
display(pd.DataFrame([xai_outputs]).T.rename(columns={0: 'path'}))
display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown(f"**Predictions used by XAI:** `{xai_run['predictions_path']}`"))
display(Markdown(f"**Prediction rows:** `{len(xai_run['predictions_df'])}`"))


## 6. XAI Metrics and Statistical Analysis

Loads the paired Grad-CAM/SHAP comparison tables, mask-ablation tables, pass-rate summaries, and optional descriptive audit breakdowns used in the report.


In [ ]:
xai_summary_audit = notebook_load_xai_committee_summary(
    cfg_xai if 'cfg_xai' in globals() else cfg,
    seed=SEED,
    split=EVAL_SPLIT,
)

display(Markdown('### Explanation Quality Comparison'))
display(Markdown('Explanation quality was assessed using continuous localization and perturbation-based '
                 'faithfulness metrics on masked attribution maps rather than binary thresholds. '
                 'Continuous metrics preserve magnitude information and avoid boundary effects '
                 'introduced by arbitrary cut-offs.'))

display(Markdown('#### Primary continuous analysis'))
if len(xai_summary_audit.get('continuous_table', [])):
    display(xai_summary_audit['continuous_table'])
    display(Markdown('*Table 5 in the report — primary continuous XAI comparison on N=120 masked attribution maps (24 per DR grade): paired Wilcoxon signed-rank + Cohen\'s $d_z$.*'))
    display(Markdown(xai_summary_audit['continuous_bottom_line_markdown']))
else:
    display(Markdown('_Continuous analysis table not found. Re-run cell 15 to generate rq_xai_continuous_*.csv._'))

display(Markdown('#### Secondary: Operational Threshold Summary (Descriptive Only)'))
display(Markdown('As a descriptive secondary analysis, we additionally report a pass rate '
                 'under a project-specific operational rule (border ratio $\\leq 0.25$ '
                 'AND $\\Delta_{k20} > 0.10$) and compare pass rates with McNemar\'s exact '
                 'test on paired binary outcomes. The thresholds of 0.25 and 0.10 are '
                 'operational choices, not literature-derived standards, which is why the '
                 'continuous analysis is treated as primary.'))
display(xai_summary_audit['summary_table'])
display(Markdown('*Table 8 in the report — threshold-based operational summary: proportion of audited targets per method that satisfy the operational rule (border ratio $\\leq 0.25$ AND $\\Delta_{k20} > 0.10$) on masked attribution maps.*'))
display(xai_summary_audit['pass_rate_fig'])
plt.close(xai_summary_audit['pass_rate_fig'])
display(Markdown('*Figure 11 in the report — descriptive companion bar chart for Table 8.*'))
display(Markdown(xai_summary_audit['bottom_line_markdown']))

In [ ]:
if not SHOW_ADVANCED_XAI_AUDIT:
    display(Markdown('_Set `SHOW_ADVANCED_XAI_AUDIT = True` to show descriptive per-class and correctness-split tables._'))
else:
    xai_advanced = notebook_load_xai_advanced_audit(
        cfg_xai if 'cfg_xai' in globals() else cfg,
        seed=SEED,
        split=EVAL_SPLIT,
        xai_outputs=xai_outputs if 'xai_outputs' in globals() else None,
    )

    display(Markdown('### Advanced Descriptive Breakdown'))

    display(Markdown('#### Correct vs Wrong Prediction Split'))
    if len(xai_advanced['correctness_view']):
        display(xai_advanced['correctness_view'])
    else:
        display(Markdown('Unavailable (table not found).'))

    display(Markdown('#### Class-wise Descriptive Explanation Metrics'))
    if len(xai_advanced['classwise_view']):
        display(xai_advanced['classwise_view'])
    else:
        display(Markdown('Unavailable (table not found).'))

    display(Markdown(f"#### Discordant Cases (n={len(xai_advanced['discord_df'])})"))
    if len(xai_advanced['discord_df']):
        display(xai_advanced['discord_df'].head(30))
    else:
        display(Markdown('Unavailable (missing rq1/rq2 columns).'))

    display(Markdown('#### Pairwise Significance'))
    display(xai_advanced['pair_df'] if len(xai_advanced['pair_df']) else pd.DataFrame({'note': ['pairwise table not found']}))

    display(Markdown('#### Runtime Status'))
    display(xai_advanced['status_df'])


## 7. Report Tables and Figures Traceability

Assembles the Grad-CAM and SHAP visual grids used in the report and records the corresponding output paths.


In [ ]:
visual_review = notebook_run_visual_review(
    cfg_xai if 'cfg_xai' in globals() else cfg,
    seed=SEED,
    split='test',
    wanted_classes=(0, 2, 3, 4),
    safe_mode=XAI_VIS_SAFE_MODE,
)

display(Markdown(
    f"**Section 8 settings:** device={visual_review['settings']['device']}, "
    f"shap_background_size={visual_review['settings']['shap_background_size']}, "
    f"shap_images={visual_review['settings']['shap_images']}"
))
display(Markdown(f"**Manifest used:** `{visual_review['manifest_path']}`"))

if visual_review['gradcam_fig'] is not None:
    display(visual_review['gradcam_fig'])
    plt.close(visual_review['gradcam_fig'])
    display(Markdown('*Figure 12 in the report — Grad-CAM heatmap examples across DR grades.*'))
else:
    display(Markdown('**Grad-CAM grid unavailable.**'))

if visual_review['shap_fig'] is not None:
    display(visual_review['shap_fig'])
    plt.close(visual_review['shap_fig'])
    display(Markdown('*Figure 13 in the report — SHAP attribution maps across DR grades.*'))
else:
    display(Markdown('**SHAP grid unavailable.**'))

display(Markdown(
    f"**Section 8 export:** dpi={visual_review['settings']['export_dpi']}, "
    f"shap_panel_size={visual_review['settings']['shap_panel_size']}, "
    f"gradcam_pair_size={visual_review['settings']['gradcam_pair_size']}"
))

if visual_review['warnings']:
    display(Markdown('### Visualization Warnings'))
    for item in visual_review['warnings']:
        display(Markdown(f'- {item}'))


## 8. Qualitative Case Study

Runs the single-case Grad-CAM, SHAP, probability, and explanation-quality workflow reported in the qualitative case study.


In [ ]:
DEMO_IMAGE_PATH = str(globals().get('DEMO_IMAGE_PATH', '')).strip()

single_case = notebook_run_single_case_report(
    cfg_xai if 'cfg_xai' in globals() else cfg,
    seed=SEED,
    image_path=DEMO_IMAGE_PATH,
    split='test',
    safe_mode=XAI_SINGLE_SAFE_MODE,
)

single_demo = single_case['single_demo']

display(Markdown(single_case['summary_markdown']))

display(Markdown('### Single-Case Probability Distribution'))
display(single_case['prob_table'])

display(Markdown(single_case['story_markdown']))

# Explanation quality profile for this single case: same four continuous metrics used
# in the aggregate Section 7 comparison, but computed on the current image only.
if 'quality_profile_markdown' in single_case:
    display(Markdown(single_case['quality_profile_markdown']))

for spec in single_case['figure_sections']:
    display(Markdown(spec['title']))
    fig_path = Path(spec['path'])
    if fig_path.exists():
        plt.figure(figsize=(16, 4.8))
        plt.imshow(plt.imread(fig_path))
        plt.axis('off')
        plt.show()
        plt.close()
    else:
        display(Markdown(f"_Figure not found: `{fig_path}`_"))
display(Markdown('*Figures 14\u201315 in the report \u2014 class-conditional Grad-CAM and per-class SHAP attribution grids for the chosen fundus image.*'))

if single_case['xai_warnings']:
    display(Markdown('### XAI Warnings'))
    for warning in single_case['xai_warnings']:
        display(Markdown(f'- {warning}'))

## 9. Audit Checklist

- Start from this notebook, `src/xai.py`, `src/data.py`, and `src/train.py`.
- Treat `src/xai_*.py` modules as implementation details reached through `src/xai.py`.
- Keep `configs/base.yaml` as the source of truth for seed, split, paths, and hyperparameters.
- Verify source parsing with `python -m py_compile src/*.py docs/app.py`.
- Verify public imports with `python -c "from src import xai, data, train; print('OK imports')"`.
- Verify this notebook remains valid JSON with `python -c "import json; json.load(open('notebooks/project_demo.ipynb')); print('OK notebook JSON')"`.
- Verify submitted headline values from saved CSV artifacts before rerunning expensive training or XAI jobs.
